# Evaluation of temporal structure

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
from mlde_utils import cp_model_rotated_pole
import numpy as np
import os
from statsmodels.tsa.stattools import acf
import string
import xarray as xr

from mlde_analysis.furflex_data import prep_eval_data
from mlde_analysis.psd import plot_psd, pysteps_rapsd
from mlde_analysis.display import pretty_table

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## Figure: ACF structure

* spatial with-in day acf per lag (1-24 hours ahead)

In [ ]:
nlags = 23

In [ ]:
# %%time

# def func(da, nlags=23):
#     # display(da)
#     return xr.apply_ufunc(
#         acf,
#         da.squeeze("date").compute(),
#         input_core_dims=[["hour"]],
#         output_core_dims=[["lag"]],
#         vectorize=True,
#         kwargs=dict(nlags=nlags),
#         # dask="allowed",
#         # dask="parallelized",
#         # dask_gufunc_kwargs={"output_sizes": {"lag": nlags+1}},
#     )

# # EVAL_DS["CPM"]["target_pr"].drop_vars(["time_period", "dec_adjusted_year", "stratum", "tp_season_year"]) \
# #     .coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(func).compute()

# acf_da = EVAL_DS["CPM"]["pred_pr"].coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(func, nlags=nlags)
# acf_da

In [ ]:
%%time

def full_dask_acf(da, nlags = 23):
    # display(da)
    return xr.apply_ufunc(
        acf,
        da.squeeze("date"),
        input_core_dims=[["hour"]],
        output_core_dims=[["lag"]],
        vectorize=True,
        kwargs=dict(nlags=nlags),
        dask="parallelized",
        dask_gufunc_kwargs={"output_sizes": 
            {
                "lag": nlags+1,
            }
        },
    )

full_dask_acf_da = EVAL_DS["CPM"]["pred_pr"].coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(full_dask_acf, nlags=nlags)
full_dask_acf_da = full_dask_acf_da.assign_coords(lag=np.arange(0,24))

In [ ]:
%%time

full_dask_target_acf_da = EVAL_DS["CPM"]["target_pr"].coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(full_dask_acf, nlags=nlags)

In [ ]:
%%time

xr.concat(
    [
        full_dask_target_acf_da.drop_sel(lag=0).mean(["date", "ensemble_member"]).expand_dims(model=["CPM"]),
        full_dask_acf_da.drop_sel(lag=0).mean(["date", "ensemble_member", "sample_id"]),
        
    ],
    dim="model",
).plot(
    col="lag",
    row="model",
    vmax=1, 
    # subplot_kws={"projection": cp_model_rotated_pole},
)
# for ax in g.axs.flat:
#     ax.coastlines()
plt.show()

## Single point ACF

In [ ]:
%%time

single_point_pred_acf_da = EVAL_DS["CPM"]["pred_pr"].isel(grid_latitude=32, grid_longitude=32).coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(full_dask_acf, nlags=nlags).drop_attrs().rename("ACF")
single_point_pred_acf_da.drop_sel(lag=0).mean(["date", "ensemble_member", "sample_id"]).plot(hue="model")

single_point_target_acf_da = EVAL_DS["CPM"]["target_pr"].isel(grid_latitude=32, grid_longitude=32).coarsen(time=24).construct(time=("date", "hour")).groupby("date").map(full_dask_acf, nlags=nlags).drop_attrs().rename("ACF")
single_point_target_acf_da.drop_sel(lag=0).mean(["date", "ensemble_member"]).plot(label="CPM", linestyle=":", color="k", alpha=0.5)

In [ ]:
client.close()